# **Exploratory Data Analysis: Understanding the Drivers of Agricultural Yield in Maji Ndogo**

<center>
<h1><b> 1. INTRODUCTION</b></h1>
</center>

Agriculture plays a critical role in ensuring food security and economic sustainability. Maximizing crop productivity requires a thorough understanding of how environmental, geographical, and soil-related factors influence agricultural yield. Before developing predictive machine learning models, it is essential to explore the underlying characteristics of the data to identify patterns, validate assumptions, and uncover relationships between variables.

This notebook presents an Exploratory Data Analysis (EDA) of the processed agricultural field and weather datasets for the fictional country of **Maji Ndogo**. The datasets have already undergone data ingestion, cleaning, preprocessing, and feature engineering through dedicated data processing pipelines, resulting in analysis-ready data.

The primary objective of this analysis is to investigate the factors associated with **`Standard_yield`**, which serves as the target variable representing standardized crop productivity. Throughout this notebook, we examine the distribution of the target variable, explore the characteristics of the predictor variables, assess relationships between features, perform statistical hypothesis testing where appropriate, and identify insights that will inform feature engineering and predictive modelling.

The findings from this exploratory analysis will provide a deeper understanding of the dataset, highlight potential drivers of crop productivity.

## 2. Importing Modules
**This section imports essential libraries for data handling, statistical analysis and visualization, along with custom project modules for configuration and processing field and weather data.**


In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
import numpy as np
import scipy as scipy
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import config_params
from src.field_data_processor import FieldDataProcessor
from src.weather_data_processor import WeatherDataProcessor

3. Data Ingestion and Processing

This section initializes custom data processing classes for field and weather data using configuration parameters. 
The process() methods handle data extraction, cleaning, and transformation, after which the processed datasets are stored as structured DataFrames for further analysis.

* `FieldDataProcessor` is used to process field-related data and store the cleaned result in `field_df`.
* `WeatherDataProcessor` processes weather data and stores the aggregated results in `weather_df` (using `mean_df`).
* Finally, `.shape` is printed for both datasets to confirm they were loaded and processed correctly.

In [ ]:
field_processor = FieldDataProcessor(config_params)
field_processor.process()
field_df = field_processor.df

weather_processor = WeatherDataProcessor(config_params)
weather_processor.process()
weather_df = weather_processor.mean_df

print(field_df.shape)
print(weather_df.shape)

In [ ]:
!pytest -v tests/test_field_data_quality.py

## 4. Exploratory Data Analysis (EDA)

### 4.1 Overview

This section explores the cleaned field and weather datasets to understand their structure, distributions, relationships, and potential patterns that may influence agricultural outcomes.

### 4.2 Data Overview

In [ ]:
field_df.head()

In [ ]:
weather_df.head()

In [ ]:
field_df.describe().round(4).drop(columns = ["Field_ID", "Weather_station"])

In [ ]:
weather_df.describe().round(4)

In [ ]:
field_df.info()

In [ ]:
std = field_df['Standard_yield'].std()
mean = round(float(field_df['Standard_yield'].mean()), 3)
skew = round(float(field_df["Standard_yield"].skew()), 3)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(field_df['Standard_yield'], bins=30, kde=True, ax=ax)
ax.axvline(mean, color='red', linestyle='dashed', linewidth=2, label=f'mean: {mean}')
ax.axvline(mean + std, color='black', linestyle='-', label='+1 std')
ax.axvline(mean - std, color='black', linestyle='-', label='-1 std')
ax.axvline(skew, color = "green", linestyle='-', label= f'skew: {skew}')
ax.set_title('Distribution of Standard_yield')
ax.legend()
plt.tight_layout()
plt.show()

**The standard yield distribution appears approximately normal. The mean (0.534) and median (0.529) are nearly identical, indicating minimal skewness. Additionally, the first and third quartiles are almost equally distant from the median, suggesting a balanced central distribution. The upper and lower tails also extend similar distances from the median (0.369 and 0.358 respectively), providing further evidence of symmetry. These observations support the bell-shaped pattern seen in the histogram and suggest that the variable is approximately normally distributed, although formal normality tests would be required to confirm this statistically**

In [ ]:
correlation= (field_df.corr(numeric_only=True)["Standard_yield"]
              .abs().sort_values(ascending=False)
              .drop(["Weather_station", "Field_ID"], 
                    errors = "ignore"))
corr_df = pd.DataFrame(correlation)
corr_df

**Correlation analysis indicates that no individual numeric feature has a strong linear relationship with Standard_yield. The highest absolute correlation is 0.286 (Pollution_level), suggesting that Standard_yield is not explained by a single dominant linear predictor. This also indicates that predictive performance may depend on the combined effects of multiple features and potentially nonlinear relationships, which can be explored using machine learning models.**